# CS690A Project 3 - SMI Pipeline v2, Phase 1: Method 3 (identity vs. program markers)
Locked baseline niches, new evidence: identity markers, program markers, ligand-receptor pairs. Output: `outputs/m3_identity_program/`.

In [1]:
# run this cell, then restart the runtime (pandas>=2.3), then run from the CONFIG cell
!pip install -q scanpy leidenalg igraph harmony-pytorch "pandas>=2.3" scikit-learn squidpy omnipath git+https://github.com/prabhakarlab/Banksy_py.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.


In [2]:
DATA_PATH = "/content/global_z0.h5ad"  # adjustable
BASELINE_LABELS_PATH = "/content/banksy_smi_labels.csv"  # adjustable
OUT_DIR = "/content/outputs/m3_identity_program"  # adjustable
METHOD_ID = "m3_identity_program"
SEED = 0  # adjustable
SMOKE_TEST = False  # adjustable
SMOKE_N = 2000  # adjustable

BASELINE_COL = "banksy"
N_NICHES_EXPECTED = 7
BANKSY_K_GEOM = 30
BANKSY_LAMBDA = 0.8
BANKSY_RES = 0.3

TOP_N = 10  # adjustable
UBIQ_CUTOFF = 0.6  # adjustable
STATE_ID_TOP = 20  # adjustable
MIN_STATE_CELLS = 50  # adjustable
MIN_UNIT_CELLS = 10  # adjustable
MIN_UNITS_SIDE = 3  # adjustable
PSEUDOBULK_MIN_UNITS = 6  # adjustable

PCA_DIMS = 50  # adjustable
SCALE_FOR_STATES = True  # adjustable
STATE_KNN = 30  # adjustable
STATE_RES_GRID = [0.5, 0.75, 1.0, 1.5, 2.0]  # adjustable
STATE_TARGET = (15, 25)  # adjustable

N_PERMS = 1000  # adjustable
LR_PVAL = 0.05  # adjustable
LR_THRESHOLD = 0.1  # adjustable
LR_TOP = 5  # adjustable
LR_TIMEOUT_S = 1800  # adjustable
LR_MIN_PCT = 0.1  # adjustable
K_SPATIAL = 30  # adjustable

N_BOOT = 5  # adjustable
BOOT_FRAC = 0.8  # adjustable

BASELINE_REF = {"n_niches": 7, "ARI": 0.248, "NMI": 0.323}
CONFIG_KEYS = [k for k in list(globals()) if k.isupper() and not k.startswith("_")]

In [3]:
import os, json, time, re, shutil, warnings, platform, random
import numpy as np, pandas as pd, scipy.sparse as sp, scipy.stats as st
import scanpy as sc, anndata
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from sklearn.neighbors import NearestNeighbors
warnings.filterwarnings("ignore")
import numba
N_JOBS = int(numba.config.NUMBA_NUM_THREADS)
sc.settings.n_jobs = N_JOBS
numba.set_num_threads(N_JOBS)

t_start = time.time()
os.makedirs(OUT_DIR, exist_ok=True)
N_PERMS_EFF = 100 if SMOKE_TEST else N_PERMS
MIN_STATE_CELLS_EFF = 10 if SMOKE_TEST else MIN_STATE_CELLS
MIN_UNIT_CELLS_EFF = 3 if SMOKE_TEST else MIN_UNIT_CELLS
MIN_UNITS_SIDE_EFF = 2 if SMOKE_TEST else MIN_UNITS_SIDE
N_BOOT_EFF = 2 if SMOKE_TEST else N_BOOT
LR_TIMEOUT_EFF = 300 if SMOKE_TEST else LR_TIMEOUT_S
LFC_EPS = 0.01
RUN = {}

def out(name): return os.path.join(OUT_DIR, name)
def _js(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return str(o)
def save_json(path, obj):
    with open(path, "w") as f: json.dump(obj, f, indent=2, default=_js)
def seed_all(s):
    np.random.seed(s); random.seed(s); torch.manual_seed(s)
def sorted_niches(vals):
    u = [str(x) for x in pd.unique(pd.Series(vals).astype(str))]
    return sorted(u, key=lambda x: (not x.isdigit(), int(x) if x.isdigit() else 0, x))
def order_df(df, niches, col="niche"):
    df = df.copy()
    df[col] = pd.Categorical(df[col].astype(str), categories=niches, ordered=True)
    df = df.sort_values([col, "rank"]).reset_index(drop=True)
    df[col] = df[col].astype(str)
    return df

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


In [4]:
# ---- utilities A: data, BANKSY, spatial kNN, states, markers, evaluation ----
def load_smi(path, smoke, smoke_n, seed):
    ad = sc.read_h5ad(path)
    ad = ad[:, ~ad.var_names.str.startswith("NegPrb")].copy()
    ad = ad[ad.obs["Region"].notna()].copy()
    if smoke:
        idx = np.sort(np.random.RandomState(seed).choice(ad.n_obs, smoke_n, replace=False))
        ad = ad[idx].copy()
    ad.X = ad.X.astype(np.float32)
    ad.layers["counts"] = ad.X.copy()
    sc.pp.normalize_total(ad, target_sum=1e4); sc.pp.log1p(ad)
    xy = np.asarray(ad.obsm["spatial_global"])
    ad.obs["cx_g"], ad.obs["cy_g"] = xy[:, 0], xy[:, 1]
    ad.obs["Region"] = ad.obs["Region"].astype(str)
    ad.obs["batch"] = ad.obs["batch"].astype(str)
    ad.obs_names = ad.obs_names.astype(str)
    return ad

def run_banksy(ad, k_geom, lam, res, seed):
    from banksy.initialize_banksy import initialize_banksy
    from banksy.embed_banksy import generate_banksy_matrix
    from banksy_utils.umap_pca import pca_umap
    from banksy.cluster_methods import run_Leiden_partition
    from harmony import harmonize
    coords = ("cx_g", "cy_g", "spatial_global")
    bank = initialize_banksy(ad, coords, k_geom, nbr_weight_decay="scaled_gaussian", max_m=1,
        plt_edge_hist=False, plt_nbr_weights=False, plt_agf_angles=False, plt_theta=False)
    bank, _ = generate_banksy_matrix(ad, bank, [lam], max_m=1)
    seed_all(seed)
    pca_umap(bank, pca_dims=[50], add_umap=False, plt_remaining_var=False)
    sub = bank["scaled_gaussian"][lam]["adata"]
    seed_all(seed)
    sub.obsm["reduced_pc_50"] = harmonize(sub.obsm["reduced_pc_50"], sub.obs, batch_key="batch")
    r, _ = run_Leiden_partition(bank, [res], num_nn=50, num_iterations=-1, partition_seed=1234, match_labels=False)
    plt.close("all")
    labels = pd.Series([str(x) for x in r.loc[r.index[0], "labels"].dense], index=sub.obs_names).reindex(ad.obs_names)
    pcs = pd.DataFrame(np.asarray(sub.obsm["reduced_pc_50"]), index=sub.obs_names).reindex(ad.obs_names).values
    return labels.values, pcs

def spatial_knn(coords, batch, k):
    n, b = len(coords), np.asarray(batch)
    rows, cols, bbox = [], [], {}
    for g in pd.unique(b):
        ix = np.where(b == g)[0]
        nn = NearestNeighbors(n_neighbors=min(k + 1, len(ix))).fit(coords[ix])
        nb = nn.kneighbors(coords[ix], return_distance=False)[:, 1:]
        rows.append(np.repeat(ix, nb.shape[1])); cols.append(ix[nb].ravel())
        bbox[g] = (coords[ix].min(0), coords[ix].max(0))
    gs = list(bbox)
    for i in range(len(gs)):
        for j in range(i + 1, len(gs)):
            lo = np.maximum(bbox[gs[i]][0], bbox[gs[j]][0]); hi = np.minimum(bbox[gs[i]][1], bbox[gs[j]][1])
            if np.all(hi > lo): print(f"WARNING: spatial_global coordinates of batches {gs[i]} and {gs[j]} overlap")
    r, c = np.concatenate(rows), np.concatenate(cols)
    return sp.csr_matrix((np.ones(len(r), dtype=np.float32), (r, c)), shape=(n, n))

def expression_pcs(ad, n_comps, seed, scale):
    a = anndata.AnnData(np.asarray(ad.X.todense()) if sp.issparse(ad.X) else ad.X.copy(), obs=ad.obs[["batch"]].copy())
    if scale: sc.pp.scale(a, max_value=10)
    sc.pp.pca(a, n_comps=n_comps, random_state=seed)
    from harmony import harmonize
    seed_all(seed)
    return np.asarray(harmonize(a.obsm["X_pca"], a.obs, batch_key="batch"))

def state_clusters(pcs, res, n_neighbors, seed):
    a = anndata.AnnData(np.zeros((pcs.shape[0], 1), dtype=np.float32))
    a.obsm["X_pca"] = np.asarray(pcs)
    sc.pp.neighbors(a, n_neighbors=n_neighbors, use_rep="X_pca", random_state=seed)
    sc.tl.leiden(a, resolution=res, flavor="igraph", n_iterations=2, directed=False, random_state=seed)
    return a.obs["leiden"].astype(str).values

def run_wilcoxon(counts, var_names, labels, obs_names):
    mk = anndata.AnnData(counts.copy(), obs=pd.DataFrame({"grp": pd.Categorical(np.asarray(labels).astype(str))}, index=obs_names),
        var=pd.DataFrame(index=var_names))
    sc.pp.normalize_total(mk, target_sum=1e4); sc.pp.log1p(mk)
    sc.tl.rank_genes_groups(mk, "grp", method="wilcoxon", use_raw=False)
    df = sc.get.rank_genes_groups_df(mk, group=None)
    df = df.rename(columns={"group": "niche", "names": "gene", "scores": "score", "logfoldchanges": "log2FC", "pvals_adj": "padj"})
    df = df.sort_values("score", ascending=False, kind="stable")
    df["rank"] = df.groupby("niche", sort=False).cumcount() + 1
    return df[["niche", "rank", "gene", "log2FC", "score", "padj"]].reset_index(drop=True)

def top10_markers(ad, labels, n):
    full = run_wilcoxon(ad.layers["counts"], ad.var_names, labels, ad.obs_names)
    niches = sorted_niches(labels)
    return order_df(full[full["rank"] <= n], niches), order_df(full, niches)

def pct_expr(counts, labels, niches, genes):
    C = counts.tocsr() if sp.issparse(counts) else sp.csr_matrix(counts)
    nz = C.copy(); nz.data = (nz.data > 0).astype(np.float32)
    codes = pd.Categorical(np.asarray(labels).astype(str), categories=niches).codes
    n = len(codes)
    M = sp.csr_matrix((np.ones(n), (codes, np.arange(n))), shape=(len(niches), n))
    inn = np.asarray((M @ nz).todense()); size = np.asarray(M.sum(1)).reshape(-1, 1)
    tot = inn.sum(0, keepdims=True)
    pin = pd.DataFrame(inn / size, index=niches, columns=genes)
    pout = pd.DataFrame((tot - inn) / np.maximum(n - size, 1), index=niches, columns=genes)
    return pin, pout

def ubiquitous_mask(pct_in, cutoff):
    return (pct_in >= cutoff).all(axis=0)

def evaluate(labels, region):
    niches = sorted_niches(labels)
    ct = pd.crosstab(np.asarray(labels).astype(str), np.asarray(region).astype(str)).reindex(niches)
    ct.index.name, ct.columns.name = "niche", "Region"
    return {"ct": ct, "majority": ct.idxmax(axis=1), "purity": ct.max(axis=1) / ct.sum(axis=1),
        "ARI": float(adjusted_rand_score(region, labels)), "NMI": float(normalized_mutual_info_score(region, labels))}

def find_unit_col(obs, min_units):
    pat = re.compile(r"^(fov|fov_id|fovs|sample|sample_id|sampleid|slide|slide_id|patient|patient_id|donor)$", re.I)
    for c in obs.columns:
        if c in ("batch", "Region", "niche"): continue
        if pat.match(c) or "fov" in c.lower():
            if min_units <= obs[c].nunique() <= len(obs) / 20: return c
    return None

In [5]:
# ---- utilities B: Wilcoxon z, pseudobulk, program markers, L-R evidence ----
def wilcoxon_z(a, b):
    n1, n2 = a.shape[0], b.shape[0]
    r = st.rankdata(np.vstack([a, b]), axis=0)
    return (r[:n1].sum(0) - n1 * (n1 + n2 + 1) / 2) / np.sqrt(n1 * n2 * (n1 + n2 + 1) / 12)

def bh(p):
    p = np.asarray(p, dtype=float); n = len(p); o = np.argsort(p)
    q = np.minimum.accumulate((p[o] * n / np.arange(1, n + 1))[::-1])[::-1]
    res = np.empty(n); res[o] = np.minimum(q, 1)
    return res

def pseudobulk(X, unit, flag, min_cells):
    uk, inv = np.unique(unit * 2 + flag, return_inverse=True)
    M = sp.csr_matrix((np.ones(len(inv)), (inv, np.arange(len(inv)))), shape=(len(uk), len(inv)))
    cnt = np.asarray(M.sum(1)).ravel()
    P = np.asarray(M @ X) / cnt[:, None]
    keep = cnt >= min_cells
    return P[keep], (uk[keep] % 2)

def program_markers(Xl, Xe, niche, state, unit, genes, excl_mask, niches, min_state, min_unit, min_side, top_n):
    rows, diag = [], []
    allowed = np.where(~excl_mask)[0]
    for nn in niches:
        inn = niche == nn
        num, den, lnum, wsum, nst = np.zeros(len(genes)), 0.0, np.zeros(len(genes)), 0.0, 0
        for s in pd.unique(state):
            sm = state == s
            n_in, n_out = int((sm & inn).sum()), int((sm & ~inn).sum())
            if n_in < min_state or n_out < min_state: continue
            ix = np.where(sm)[0]; fl = inn[ix].astype(int)
            if unit is None:
                a, b = Xl[ix[fl == 1]], Xl[ix[fl == 0]]
            else:
                P, f = pseudobulk(Xl[ix], unit[ix], fl, min_unit)
                if (f == 1).sum() < min_side or (f == 0).sum() < min_side: continue
                a, b = P[f == 1], P[f == 0]
            w = np.sqrt(min(n_in, n_out))
            lfc = np.log2((Xe[ix[fl == 1]].mean(0) + LFC_EPS) / (Xe[ix[fl == 0]].mean(0) + LFC_EPS))
            num += w * wilcoxon_z(a, b); den += w * w; lnum += w * lfc; wsum += w; nst += 1
        diag.append({"niche": nn, "n_states_used": nst, "de_unit": "cell" if unit is None else "pseudobulk"})
        if nst == 0: continue
        z = num / np.sqrt(den); p = 2 * st.norm.sf(np.abs(z))
        padj = np.full(len(genes), np.nan); padj[allowed] = bh(p[allowed])
        order = allowed[np.argsort(-z[allowed])]
        order = [g for g in order if z[g] > 0][:top_n]
        for r, g in enumerate(order, 1):
            rows.append({"niche": nn, "rank": r, "gene": genes[g], "z": z[g], "padj": padj[g], "log2FC": lnum[g] / wsum, "n_states": nst})
    cols = ["niche", "rank", "gene", "z", "padj", "log2FC", "n_states"]
    return pd.DataFrame(rows, columns=cols), pd.DataFrame(diag)

def get_lr_pairs(panel):
    import omnipath as op
    df = op.interactions.import_intercell_network(transmitter_params={"categories": "ligand"}, receiver_params={"categories": "receptor"})
    for a, b in [("genesymbol_intercell_source", "genesymbol_intercell_target"), ("source_genesymbol", "target_genesymbol")]:
        if a in df.columns and b in df.columns: break
    else:
        raise KeyError(f"no gene symbol columns in {list(df.columns)}")
    p = df[[a, b]].dropna(); p.columns = ["source", "target"]
    ps = set(panel)
    return p[p.source.isin(ps) & p.target.isin(ps)].drop_duplicates().reset_index(drop=True)

def _res_get(res, k):
    return res[k] if isinstance(res, dict) else getattr(res, k)

def ligrec_evidence(ad, pairs, n_perms, seed, pval, thr, top, timeout):
    import squidpy as sq
    from concurrent.futures import ThreadPoolExecutor
    ex = ThreadPoolExecutor(max_workers=1)
    fut = ex.submit(lambda: sq.gr.ligrec(ad, cluster_key="niche", n_perms=n_perms, interactions=pairs,
        corr_method="fdr_bh", threshold=thr, use_raw=False, copy=True, seed=seed))
    try: res = fut.result(timeout=timeout)
    finally: ex.shutdown(wait=False, cancel_futures=True)
    means, pv = _res_get(res, "means"), _res_get(res, "pvalues")
    parts = []
    for c in means.columns:
        d = pd.DataFrame({"mean_score": means[c].values, "pval": pv[c].values}, index=means.index)
        d = d[(d["pval"] < pval) & d["mean_score"].notna()]
        parts.append(pd.DataFrame({"niche": str(c[1]), "ligand": d.index.get_level_values(0), "receptor": d.index.get_level_values(1),
            "source_niche": str(c[0]), "mean_score": d["mean_score"].values, "pval": d["pval"].values}))
    df = pd.concat(parts, ignore_index=True)
    return df.sort_values("mean_score", ascending=False).groupby("niche", sort=False).head(top)

def custom_lr(Xl, genes, niche, niches, knn, pairs, top, min_pct):
    gi = {g: i for i, g in enumerate(genes)}
    li, ri = pairs["source"].map(gi).values, pairs["target"].map(gi).values
    rows = []
    for r in niches:
        inn = np.where(niche == r)[0]
        nb = np.unique(knn[inn].indices); nb = nb[niche[nb] != r]
        if len(nb) == 0: continue
        Xr, Xo = Xl[inn], Xl[nb]
        rmean, rpct, lmean, lpct = Xr.mean(0), (Xr > 0).mean(0), Xo.mean(0), (Xo > 0).mean(0)
        score = rmean[ri] * lmean[li]
        ok = (rpct[ri] >= min_pct) & (lpct[li] >= min_pct)
        if not ok.any(): continue
        sel = np.where(ok)[0]; sel = sel[np.argsort(-score[sel])][:top]
        src = niche[nb]
        gm = {s: Xo[src == s].mean(0) for s in niches if s != r and (src == s).any()}
        for k in sel:
            best = max(gm, key=lambda s: gm[s][li[k]])
            rows.append({"niche": r, "ligand": pairs["source"].iat[k], "receptor": pairs["target"].iat[k],
                "source_niche": best, "mean_score": float(score[k]), "pval": np.nan})
    return pd.DataFrame(rows, columns=["niche", "ligand", "receptor", "source_niche", "mean_score", "pval"])

def identity_boot(counts, genes, labels, obs_names, keep, ref, top_n, n_boot, frac, seed):
    rs = np.random.RandomState(seed); n = len(labels); niches = sorted_niches(labels); rows = []
    for b in range(n_boot):
        idx = np.sort(rs.choice(n, int(frac * n), replace=False))
        f = run_wilcoxon(counts[idx], genes, np.asarray(labels)[idx], obs_names[idx])
        f = f[f["gene"].map(keep).fillna(False).astype(bool)]
        f = f.sort_values("score", ascending=False, kind="stable")
        for nn in niches:
            cur = set(f[f["niche"] == nn]["gene"].head(top_n)); base = ref[nn]
            rows.append({"niche": nn, "boot": b, "jaccard": len(cur & base) / max(len(cur | base), 1)})
    return pd.DataFrame(rows)

In [6]:
# ---- utilities C: contract writers ----
def save_labels(ad, niche, state=None):
    lab = pd.DataFrame({"niche": np.asarray(niche).astype(str), "Region": ad.obs["Region"].values,
        "cx_g": ad.obs["cx_g"].values, "cy_g": ad.obs["cy_g"].values, "batch": ad.obs["batch"].values}, index=ad.obs_names)
    if state is not None: lab["state"] = np.asarray(state).astype(str)
    lab.index.name = "cell_id"
    lab.to_csv(out("labels.csv"))

def build_summary(niche, niches, ev, stab):
    n = pd.Series(np.asarray(niche).astype(str)).value_counts().reindex(niches)
    df = pd.DataFrame({"niche": niches, "n_cells": n.values, "frac_cells": n.values / len(niche),
        "majority_region": ev["majority"].reindex(niches).values, "purity": ev["purity"].reindex(niches).values})
    if stab is not None: df["stability"] = stab.reindex(niches).values
    return df

def plot_niche_vs_truth(ad, niche, path):
    fig, axs = plt.subplots(1, 2, figsize=(18, 8))
    x, y = ad.obs["cx_g"].values, ad.obs["cy_g"].values
    for ax, vals, title in zip(axs, [ad.obs["Region"].values, np.asarray(niche).astype(str)], ["expert region", "niche"]):
        cats = sorted_niches(vals); cmap = plt.get_cmap("tab10" if len(cats) <= 10 else "tab20")
        for i, c in enumerate(cats):
            m = vals == c
            ax.scatter(x[m], y[m], s=1, color=cmap(i % cmap.N), label=f"{c} ({m.sum()})", rasterized=True)
        ax.legend(markerscale=8, fontsize=7, loc="best"); ax.set_title(title); ax.set_aspect("equal")
    fig.savefig(path, dpi=150, bbox_inches="tight"); plt.close(fig)

def save_contract(ad, niche, state, top, ev, stab, metrics, run_cfg):
    niches = sorted_niches(niche)
    save_labels(ad, niche, state)
    top.to_csv(out("niche_markers_long.csv"), index=False)
    top.pivot(index="rank", columns="niche", values="gene").reindex(columns=niches).to_csv(out("niche_markers_wide.csv"))
    build_summary(niche, niches, ev, stab).to_csv(out("niche_summary.csv"), index=False)
    ev["ct"].to_csv(out("crosstab_vs_region.csv"))
    save_json(out("metrics.json"), metrics); save_json(out("run_config.json"), run_cfg)
    plot_niche_vs_truth(ad, niche, out("niche_vs_truth.png"))

In [7]:
seed_all(SEED)
ad = load_smi(DATA_PATH, SMOKE_TEST, SMOKE_N, SEED)
print(ad)
print("obs columns:", list(ad.obs.columns))
assert "batch" in ad.obs and "Region" in ad.obs and "spatial_global" in ad.obsm
genes = np.array(ad.var_names)
counts = ad.layers["counts"]
Xl = np.asarray(ad.X.todense(), dtype=np.float32) if sp.issparse(ad.X) else np.asarray(ad.X, dtype=np.float32)
Xe = np.expm1(Xl)
coords = ad.obs[["cx_g", "cy_g"]].values

AnnData object with n_obs × n_vars = 18895 × 960
    obs: 'cell_type', 'niche', 'cx', 'cy', 'cx_g', 'cy_g', 'merge_cell_type', 'celltype_refined', 'batch', 'region', 'Region'
    uns: 'log1p'
    obsm: 'spatial', 'spatial_global'
    layers: None (.X), 'counts'
obs columns: ['cell_type', 'niche', 'cx', 'cy', 'cx_g', 'cy_g', 'merge_cell_type', 'celltype_refined', 'batch', 'region', 'Region']


In [8]:
if os.path.exists(BASELINE_LABELS_PATH):
    bl = pd.read_csv(BASELINE_LABELS_PATH, index_col=0); bl.index = bl.index.astype(str)
    assert BASELINE_COL in bl.columns, f"column {BASELINE_COL} missing: {list(bl.columns)}"
    s = bl[BASELINE_COL].reindex(ad.obs_names)
    assert s.notna().all(), f"{int(s.isna().sum())} cells missing from baseline labels"
    niche = s.astype(str).values
    RUN["labels_source"] = f"loaded:{BASELINE_LABELS_PATH}"
else:
    niche, _ = run_banksy(ad, BANKSY_K_GEOM, BANKSY_LAMBDA, BANKSY_RES, SEED)
    RUN["labels_source"] = f"recomputed:k_geom={BANKSY_K_GEOM},lambda={BANKSY_LAMBDA},res={BANKSY_RES},seed={SEED}"
niches = sorted_niches(niche)
if SMOKE_TEST:
    print(f"smoke test: {len(niches)} niches present (7 not enforced)")
else:
    assert len(niches) == N_NICHES_EXPECTED, f"expected {N_NICHES_EXPECTED} niches, got {len(niches)}"
ad.obs["niche"] = pd.Categorical(niche, categories=niches)
niche = np.asarray(niche).astype(str)
save_labels(ad, niche)
print(RUN["labels_source"], pd.Series(niche).value_counts().reindex(niches).to_dict())

loaded:/content/banksy_smi_labels.csv {'0': 5665, '1': 4764, '2': 3763, '3': 3122, '4': 900, '5': 522, '6': 159}


In [9]:
std_top, std_full = top10_markers(ad, niche, TOP_N)
std_top.to_csv(out("niche_markers_long.csv"), index=False)
std_top.pivot(index="rank", columns="niche", values="gene").reindex(columns=niches).to_csv(out("niche_markers_wide.csv"))
ev = evaluate(niche, ad.obs["Region"].values)
ev["ct"].to_csv(out("crosstab_vs_region.csv"))
build_summary(niche, niches, ev, None).to_csv(out("niche_summary.csv"), index=False)
print(f"reporting only: n_niches={len(niches)} ARI={ev['ARI']:.3f} NMI={ev['NMI']:.3f} (baseline ref ARI {BASELINE_REF['ARI']}, NMI {BASELINE_REF['NMI']})")
print(pd.DataFrame({"majority_region": ev["majority"], "purity": ev["purity"].round(2)}))

reporting only: n_niches=7 ARI=0.248 NMI=0.323 (baseline ref ARI 0.248, NMI 0.323)
               majority_region  purity
niche                                 
0      myeloid-enriched stroma    0.78
1                       stroma    0.55
2                       stroma    0.53
3               tumor interior    0.61
4      myeloid-enriched stroma    0.68
5                       stroma    0.46
6      myeloid-enriched stroma    0.68


In [10]:
pct_in, pct_out = pct_expr(counts, niche, niches, genes)
ubiq = ubiquitous_mask(pct_in, UBIQ_CUTOFF)
idf = std_full[std_full["gene"].map(~ubiq).astype(bool)].sort_values("score", ascending=False, kind="stable")
idf["rank"] = idf.groupby("niche", sort=False).cumcount() + 1
idf = order_df(idf[idf["rank"] <= TOP_N], niches)
idf["pct_in"] = [pct_in.at[n, g] for n, g in zip(idf["niche"], idf["gene"])]
idf["pct_out"] = [pct_out.at[n, g] for n, g in zip(idf["niche"], idf["gene"])]
idf.to_csv(out("identity_markers.csv"), index=False)

ubq_rows = []
for n in niches:
    top_n_genes = std_top[std_top["niche"] == n]["gene"].tolist()
    removed = [g for g in top_n_genes if ubiq[g]]
    ubq_rows.append({"niche": n, "n_ubiquitous_genes_total": int(ubiq.sum()), "n_removed_from_top_list": len(removed), "removed_genes": ";".join(removed)})
ubq = pd.DataFrame(ubq_rows)
ubq.to_csv(out("ubiquity_filter.csv"), index=False)
print(f"ubiquity filter (>= {UBIQ_CUTOFF:.0%} cells in every niche) flags {int(ubiq.sum())} of {len(genes)} genes")
print(ubq[["niche", "n_removed_from_top_list", "removed_genes"]].to_string(index=False))

ubiquity filter (>= 60% cells in every niche) flags 1 of 960 genes
niche  n_removed_from_top_list removed_genes
    0                        1         MZT2A
    1                        0              
    2                        0              
    3                        0              
    4                        1         MZT2A
    5                        1         MZT2A
    6                        0              


In [11]:
pcs = expression_pcs(ad, PCA_DIMS, SEED, SCALE_FOR_STATES)
cand, cand_labels = {}, {}
for r in STATE_RES_GRID:
    cand_labels[r] = state_clusters(pcs, r, STATE_KNN, SEED); cand[r] = int(len(np.unique(cand_labels[r])))
lo, hi = STATE_TARGET; mid = (lo + hi) / 2
inrange = [r for r in STATE_RES_GRID if lo <= cand[r] <= hi]
state_res = min(inrange or STATE_RES_GRID, key=lambda r: abs(cand[r] - mid))
state = cand_labels[state_res]
pd.DataFrame({"resolution": list(cand), "n_states": list(cand.values())}).to_csv(out("state_resolution_curve.csv"), index=False)
print("state candidates:", cand, "-> chosen resolution", state_res, "with", cand[state_res], "states")
save_labels(ad, niche, state)

sid_full = run_wilcoxon(counts, ad.var_names, state, ad.obs_names).rename(columns={"niche": "state"})
sid = sid_full[sid_full["rank"] <= STATE_ID_TOP]
sid.to_csv(out("state_identity_top20.csv"), index=False)
excl_mask = np.isin(genes, sid["gene"].unique())
print("genes excluded from program markers (state identity top-%d):" % STATE_ID_TOP, int(excl_mask.sum()))

	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
Reach convergence after 2 iteration(s).
state candidates: {0.5: 5, 0.75: 8, 1.0: 10, 1.5: 12, 2.0: 15} -> chosen resolution 2.0 with 15 states
genes excluded from program markers (state identity top-20): 206


In [12]:
unit_col = find_unit_col(ad.obs, PSEUDOBULK_MIN_UNITS)
unit = pd.factorize(ad.obs[unit_col].astype(str))[0] if unit_col else None
de_unit = f"pseudobulk by {unit_col}" if unit_col else "cell-level (no FOV/sample column with enough units; cell-level DE is anti-conservative)"
print("program DE unit:", de_unit)
prog, prog_diag = program_markers(Xl, Xe, niche, state, unit, genes, excl_mask, niches,
    MIN_STATE_CELLS_EFF, MIN_UNIT_CELLS_EFF, MIN_UNITS_SIDE_EFF, TOP_N)
prog = order_df(prog, niches) if len(prog) else prog
prog.to_csv(out("program_markers.csv"), index=False)
prog_diag.to_csv(out("program_diagnostics.csv"), index=False)
print(prog_diag.to_string(index=False))

program DE unit: cell-level (no FOV/sample column with enough units; cell-level DE is anti-conservative)
niche  n_states_used de_unit
    0             14    cell
    1             14    cell
    2             13    cell
    3             10    cell
    4              7    cell
    5              4    cell
    6              0    cell


In [13]:
LR_STATUS = {}
lr = None
try:
    pairs = get_lr_pairs(genes); LR_STATUS["n_pairs_in_panel"] = int(len(pairs))
except Exception as e:
    pairs = None; LR_STATUS["pairs_error"] = repr(e)[:300]
if pairs is not None and len(pairs):
    try:
        lr = ligrec_evidence(ad, pairs, N_PERMS_EFF, SEED, LR_PVAL, LR_THRESHOLD, LR_TOP, LR_TIMEOUT_EFF)
        LR_STATUS["method"] = "squidpy_ligrec"
    except Exception as e:
        LR_STATUS["ligrec_error"] = repr(e)[:300]
    if lr is None:
        knn = spatial_knn(coords, ad.obs["batch"].values, K_SPATIAL)
        lr = custom_lr(Xl, genes, niche, niches, knn, pairs, LR_TOP, LR_MIN_PCT)
        LR_STATUS["method"] = "custom_neighbour_fallback"
if lr is None:
    lr = pd.DataFrame(columns=["niche", "ligand", "receptor", "source_niche", "mean_score", "pval"]); LR_STATUS["method"] = "none"
lr["niche"] = lr["niche"].astype(str)
lr = lr.assign(rank=lr.groupby("niche").cumcount() + 1) if len(lr) else lr.assign(rank=[])
lr = order_df(lr, niches).drop(columns="rank")
lr.to_csv(out("niche_LR.csv"), index=False)
print(LR_STATUS); print(lr.groupby("niche").size().reindex(niches).fillna(0).astype(int).to_dict())

0.00B [00:00, ?B/s]

0.00B [00:00, ?B/s]

0.00B [00:00, ?B/s]

  0%|          | 0/1000 [00:00<?, ?permutation/s]

{'n_pairs_in_panel': 2277, 'method': 'squidpy_ligrec'}
{'0': 5, '1': 5, '2': 5, '3': 5, '4': 5, '5': 5, '6': 5}


In [14]:
ref = {n: set(idf[idf["niche"] == n]["gene"]) for n in niches}
boot = identity_boot(counts, ad.var_names, niche, np.asarray(ad.obs_names), ~ubiq, ref, TOP_N, N_BOOT_EFF, BOOT_FRAC, SEED)
boot.to_csv(out("identity_bootstrap.csv"), index=False)
stab = boot.groupby("niche")["jaccard"].mean().reindex(niches)
print("identity-marker bootstrap Jaccard:", stab.round(2).to_dict())

identity-marker bootstrap Jaccard: {'0': 1.0, '1': 0.93, '2': 0.89, '3': 1.0, '4': 0.73, '5': 0.79, '6': 0.73}


In [15]:
sil = float(silhouette_score(pcs, niche, sample_size=min(5000, len(niche)), random_state=SEED)) if len(niches) > 1 else None
versions = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "scanpy": sc.__version__,
    "anndata": anndata.__version__, "torch": torch.__version__}
try:
    import squidpy; versions["squidpy"] = squidpy.__version__
except Exception: pass
metrics = {"n_niches": len(niches), "ARI": ev["ARI"], "NMI": ev["NMI"], "silhouette_expr_harmony_pcs": sil,
    "stability": {"identity_marker_bootstrap_jaccard_mean": float(stab.mean()), "per_niche": stab.to_dict(), "n_boot": N_BOOT_EFF, "boot_frac": BOOT_FRAC},
    "seed_robustness": "not applicable: niche labels are locked; bootstrap stability above covers the evidence",
    "label_free_selection": {"niche_labels": "locked baseline, nothing selected", "state_resolution_candidates_n_states": cand,
        "state_resolution_chosen": state_res, "rule": f"n_states closest to {mid} within {STATE_TARGET}; no Region used"},
    "baseline_reference": BASELINE_REF,
    "ubiquity_filter": ubq.drop(columns="removed_genes").to_dict("records"),
    "program_diagnostics": prog_diag.to_dict("records"), "ligand_receptor": LR_STATUS,
    "note": "ARI/NMI/purity are reporting only; nothing was selected using Region"}
run_cfg = {"config": {k: globals()[k] for k in CONFIG_KEYS}, "effective_smoke_adjusted": {"N_PERMS": N_PERMS_EFF,
    "MIN_STATE_CELLS": MIN_STATE_CELLS_EFF, "MIN_UNIT_CELLS": MIN_UNIT_CELLS_EFF, "MIN_UNITS_SIDE": MIN_UNITS_SIDE_EFF,
    "N_BOOT": N_BOOT_EFF, "LR_TIMEOUT_S": LR_TIMEOUT_EFF}, "labels_source": RUN["labels_source"], "de_unit": de_unit,
    "unit_column": unit_col, "n_cells": int(ad.n_obs), "n_genes": int(ad.n_vars), "versions": versions,
    "runtime_s": round(time.time() - t_start, 1)}
save_contract(ad, niche, state, std_top, ev, stab, metrics, run_cfg)

required = ["labels.csv", "niche_markers_long.csv", "niche_markers_wide.csv", "niche_summary.csv", "crosstab_vs_region.csv",
    "metrics.json", "run_config.json", "niche_vs_truth.png", "identity_markers.csv", "program_markers.csv", "niche_LR.csv"]
missing = [f for f in required if not os.path.exists(out(f))]
assert not missing, f"missing outputs: {missing}"
print("all contract files written:", sorted(os.listdir(OUT_DIR)))
print(build_summary(niche, niches, ev, stab).round(3).to_string(index=False))

all contract files written: ['crosstab_vs_region.csv', 'identity_bootstrap.csv', 'identity_markers.csv', 'labels.csv', 'metrics.json', 'niche_LR.csv', 'niche_markers_long.csv', 'niche_markers_wide.csv', 'niche_summary.csv', 'niche_vs_truth.png', 'program_diagnostics.csv', 'program_markers.csv', 'run_config.json', 'state_identity_top20.csv', 'state_resolution_curve.csv', 'ubiquity_filter.csv']
niche  n_cells  frac_cells         majority_region  purity  stability
    0     5665       0.300 myeloid-enriched stroma   0.776      1.000
    1     4764       0.252                  stroma   0.553      0.927
    2     3763       0.199                  stroma   0.526      0.891
    3     3122       0.165          tumor interior   0.609      1.000
    4      900       0.048 myeloid-enriched stroma   0.682      0.727
    5      522       0.028                  stroma   0.460      0.788
    6      159       0.008 myeloid-enriched stroma   0.679      0.727


In [16]:
shutil.make_archive("/content/m3_identity_program", "zip", OUT_DIR)
try:
    from google.colab import files
    files.download("/content/m3_identity_program.zip")
except ImportError:
    print("not in Colab; zip at /content/m3_identity_program.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>